In [ ]:
import random

ROWS, COLS = 10, 10
START = (0, 0)
GOAL = (9, 9)

OBSTACLES = {
    (1, 2), (2, 2), (3, 2),
    (4, 4), (5, 4), (6, 4),
    (7, 6), (8, 6)
}

N = 50
Pm = 0.1
Pc = 0.8
G = 100

MOVES = [
    (-1, 0),
    (1, 0),
    (0, -1),
    (0, 1)
]

def generate_individual():
    return [random.choice(MOVES) for _ in range(18)]

def fitness(individual):
    position = START
    penalty = 0

    for move in individual:
        new_position = (
            position[0] + move[0],
            position[1] + move[1]
        )

        if not (0 <= new_position[0] < ROWS and
                0 <= new_position[1] < COLS):
            penalty += 20
            continue

        if new_position in OBSTACLES:
            penalty += 50
        else:
            position = new_position

    goal_distance = (
        abs(GOAL[0] - position[0]) +
        abs(GOAL[1] - position[1])
    )

    return 1 / (1 + goal_distance + penalty)

def selection(population):
    return random.choices(
        population,
        weights=[fitness(x) for x in population],
        k=N
    )

def crossover(parent1, parent2):
    point = random.randint(1, len(parent1) - 1)

    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]

    return child1, child2

def mutate(individual):
    index = random.randint(0, len(individual) - 1)
    individual[index] = random.choice(MOVES)

population = [generate_individual() for _ in range(N)]

best_individual = max(population, key=fitness)
best_fitness = fitness(best_individual)

for generation in range(G):

    selected = selection(population)
    offspring = []

    for i in range(0, N, 2):
        parent1 = selected[i]
        parent2 = selected[(i + 1) % N]

        if random.random() <= Pc:
            child1, child2 = crossover(parent1, parent2)
        else:
            child1 = parent1[:]
            child2 = parent2[:]

        offspring.append(child1)
        offspring.append(child2)

    for individual in offspring:
        if random.random() <= Pm:
            mutate(individual)

    population = offspring[:N]

    current_best = max(population, key=fitness)

    if fitness(current_best) > best_fitness:
        best_individual = current_best[:]
        best_fitness = fitness(current_best)

print("Best Fitness:", best_fitness)

position = START
path = [position]

for move in best_individual:
    new_position = (
        position[0] + move[0],
        position[1] + move[1]
    )

    if (0 <= new_position[0] < ROWS and
        0 <= new_position[1] < COLS and
        new_position not in OBSTACLES):

        position = new_position

    path.append(position)

print("Robot Path:")
print(path)

print("\nGrid:")

for i in range(ROWS):
    for j in range(COLS):

        if (i, j) == START:
            print("S", end=" ")

        elif (i, j) == GOAL:
            print("G", end=" ")

        elif (i, j) in OBSTACLES:
            print("#", end=" ")

        elif (i, j) in path:
            print("*", end=" ")

        else:
            print(".", end=" ")

    print()

Best Fitness: 0.3333333333333333
Robot Path:
[(0, 0), (1, 0), (0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (1, 5), (2, 5), (3, 5), (3, 6), (3, 7), (3, 8), (4, 8), (5, 8), (6, 8), (6, 9), (7, 9)]

Grid:
S * * * * . . . . . 
* . # . * * . . . . 
. . # . . * . . . . 
. . # . . * * * * . 
. . . . # . . . * . 
. . . . # . . . * . 
. . . . # . . . * * 
. . . . . . # . . * 
. . . . . . # . . . 
. . . . . . . . . G 


In [ ]:
import random
import math

ROWS = 10
COLS = 10

START = (0, 0)
GOAL = (9, 9)

OBSTACLES = {
    (1, 2), (2, 2), (3, 2),
    (4, 4), (5, 4), (6, 4),
    (7, 6), (8, 6)
}

POPULATION_SIZE = 50
GENERATIONS = 100

PC_MAX = 0.9
PC_MIN = 0.5
PM_MAX = 0.3
PM_MIN = 0.05

PATH_LENGTH = 20

MOVES = [
    (-1, 0),
    (1, 0),
    (0, -1),
    (0, 1)
]


def generate_path():
    return [random.choice(MOVES) for _ in range(PATH_LENGTH)]


def get_path(chromosome, start=START):
    position = start
    path = [position]

    for move in chromosome:
        new_position = (
            position[0] + move[0],
            position[1] + move[1]
        )

        if (0 <= new_position[0] < ROWS and
            0 <= new_position[1] < COLS):

            position = new_position

        path.append(position)

    return path


def fitness(chromosome, start=START):
    path = get_path(chromosome, start)

    length = 0
    safety = 0
    smoothness = 0

    for i in range(1, len(path)):
        length += 1

        if path[i] in OBSTACLES:
            safety += 100

    for i in range(2, len(chromosome)):
        if chromosome[i] != chromosome[i - 1]:
            smoothness += 1

    final_position = path[-1]

    goal_distance = (
        abs(GOAL[0] - final_position[0]) +
        abs(GOAL[1] - final_position[1])
    )

    cost = (
        length +
        5 * smoothness +
        20 * goal_distance +
        100 * safety
    )

    return cost


def selection(population):
    population.sort(key=fitness)

    return population[:POPULATION_SIZE // 2]


def crossover(parent1, parent2):
    point = random.randint(1, PATH_LENGTH - 1)

    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]

    return child1, child2


def mutate(path, mutation_rate):
    for i in range(len(path)):
        if random.random() < mutation_rate:
            path[i] = random.choice(MOVES)

    return path


def genetic_algorithm(start=START):

    population = [
        generate_path()
        for _ in range(POPULATION_SIZE)
    ]

    best = min(population, key=fitness)
    best_cost = fitness(best)

    for generation in range(GENERATIONS):

        selected = selection(population)

        progress = generation / GENERATIONS

        crossover_rate = PC_MAX - (
            PC_MAX - PC_MIN
        ) * progress

        mutation_rate = PM_MIN + (
            PM_MAX - PM_MIN
        ) * progress

        new_population = []

        while len(new_population) < POPULATION_SIZE:

            parent1 = random.choice(selected)
            parent2 = random.choice(selected)

            if random.random() < crossover_rate:

                child1, child2 = crossover(
                    parent1, parent2
                )

            else:
                child1 = parent1[:]
                child2 = parent2[:]

            child1 = mutate(child1, mutation_rate)
            child2 = mutate(child2, mutation_rate)

            new_population.append(child1)
            new_population.append(child2)

        population = new_population[:POPULATION_SIZE]

        current_best = min(
            population,
            key=fitness
        )

        current_cost = fitness(current_best)

        if current_cost < best_cost:

            best = current_best[:]
            best_cost = current_cost

    return best, best_cost


best_path, best_cost = genetic_algorithm()

path = get_path(best_path)

print("Best Path Cost:", best_cost)
print("Robot Path:")
print(path)

print("\nGrid:")

for i in range(ROWS):

    for j in range(COLS):

        if (i, j) == START:
            print("S", end=" ")

        elif (i, j) == GOAL:
            print("G", end=" ")

        elif (i, j) in OBSTACLES:
            print("#", end=" ")

        elif (i, j) in path:
            print("*", end=" ")

        else:
            print(".", end=" ")

    print()

Best Path Cost: 75
Robot Path:
[(0, 0), (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (4, 0), (5, 0), (6, 0), (7, 0), (7, 1), (7, 2), (8, 2), (8, 3), (8, 4), (8, 5), (9, 5), (9, 5), (9, 6), (9, 7), (9, 8)]

Grid:
S . . . . . . . . . 
* . # . . . . . . . 
* . # . . . . . . . 
* . # . . . . . . . 
* . . . # . . . . . 
* . . . # . . . . . 
* . . . # . . . . . 
* * * . . . # . . . 
. . * * * * # . . . 
. . . . . * * * * G 
